# Base de casos 

Este notebook transforma la salida del cuestionario en **tres salidas principales**:

1. **`user_profiles.csv`**: una fila por usuario con su perfil.
2. **`image_responses_long.csv`**: una fila por interacción usuario–imagen.
3. **`case_base_double_partial.csv`**: base de casos doble parcial, uniendo perfil + respuesta por imagen.



Dos niveles de información:

- **Nivel 1: perfil del usuario**  
  edad, estudios, ocupación, conocimiento en IA, conocimiento del dominio, preferencias de formato y objetivos.

- **Nivel 2: interacción con cada imagen**  
  opción elegida para la explicación y valoración de satisfacción, confianza y comprensión.

Por eso conviene construir dos tablas enlazadas por `user_id` y después una tabla unificada de casos.


## Esquema resultante

### A. Base de perfiles (`user_profiles.csv`)
Una fila por persona.

**Clave**: `user_id`

### B. Base de interacciones (`image_responses_long.csv`)
Una fila por cada par `(user_id, image_id)`.

**Claves**: `user_id`, `image_id`

### C. Base de casos doble (`case_base_double_partial.csv`)
Une A y B, dejando una fila por caso candidato para CBR.

**Clave sugerida**: `case_id = user_id + image_id`

> Con 27 usuarios y 11 imágenes, el resultado esperado es **297 casos**.


In [5]:

from pathlib import Path
import pandas as pd
import numpy as np
import re
import json

INPUT_XLSX = Path("Estudio sobre explicaciones de IA en clasificación de imágenes (XAI)  (respuestas).xlsx")
OUTPUT_DIR = Path("cbr_case_base_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

raw = pd.read_excel(INPUT_XLSX)
raw.shape


(27, 57)

## 1. Inspección básica

In [6]:

print(f"Filas (usuarios): {raw.shape[0]}")
print(f"Columnas: {raw.shape[1]}")
print("\nPrimeras columnas:")
for i, c in enumerate(raw.columns[:12], start=1):
    print(i, c)


Filas (usuarios): 27
Columnas: 57

Primeras columnas:
1 Marca temporal
2 Edad
3 Nivel de estudios terminados
4 ¿A qué te dedicas actualmente? 
5 Conocimiento en IA (Inteligencia Artificial) (1 = nada, 5 = experto) 
6 ¿Cuál es tu nivel de conocimiento sobre animales en entornos salvajes?
7 Imagen 1 de 10 
8 ¿Qué tan satisfecho/a estás con la explicación mostrada? 
9 Después de ver la explicación, ¿cuánta confianza tienes en la respuesta del sistema? 
10 ¿La explicación te ayudó a entender la decisión del sistema? 
11 Imagen 2 de 10 
12 ¿Qué tan satisfecho/a estás con la explicación mostrada?  2


## 2. Funciones auxiliares

In [7]:

def extract_numeric_prefix(text):
    """Extrae el número inicial de respuestas tipo '4 Nivel técnico: ...'"""
    if pd.isna(text):
        return np.nan
    m = re.match(r"\s*(\d+)", str(text))
    return int(m.group(1)) if m else np.nan

def extract_label_after_score(text):
    """Devuelve el texto sin el prefijo numérico inicial."""
    if pd.isna(text):
        return text
    cleaned = re.sub(r"^\s*\d+\s*", "", str(text)).strip()
    return cleaned if cleaned else text

def split_multi(value):
    """Convierte campos multiselección en listas limpias."""
    if pd.isna(value):
        return []
    return [x.strip() for x in str(value).split(",") if x.strip()]

def normalize_whitespace(text):
    if pd.isna(text):
        return text
    return re.sub(r"\s+", " ", str(text)).strip()

def safe_json_list(value):
    return json.dumps(split_multi(value), ensure_ascii=False)


## 3. Construir la base de perfiles de usuario

Estas columnas salen del bloque inicial y del bloque final del formulario.


In [8]:

profile_df = pd.DataFrame({
    "user_id": [f"U{idx:03d}" for idx in range(1, len(raw) + 1)],
    "timestamp": raw["Marca temporal"],
    "age_range": raw["Edad"].map(normalize_whitespace),
    "education_level": raw["Nivel de estudios terminados"].map(normalize_whitespace),
    "occupation_raw": raw["¿A qué te dedicas actualmente? "].map(normalize_whitespace),
    "occupation_list": raw["¿A qué te dedicas actualmente? "].apply(split_multi).apply(lambda x: json.dumps(x, ensure_ascii=False)),
    "ai_knowledge_raw": raw["Conocimiento en IA (Inteligencia Artificial) (1 = nada, 5 = experto) "].map(normalize_whitespace),
    "ai_knowledge_level": raw["Conocimiento en IA (Inteligencia Artificial) (1 = nada, 5 = experto) "].apply(extract_numeric_prefix),
    "ai_knowledge_desc": raw["Conocimiento en IA (Inteligencia Artificial) (1 = nada, 5 = experto) "].apply(extract_label_after_score),
    "domain_knowledge_raw": raw["¿Cuál es tu nivel de conocimiento sobre animales en entornos salvajes?"].map(normalize_whitespace),
    "domain_knowledge_level": raw["¿Cuál es tu nivel de conocimiento sobre animales en entornos salvajes?"].apply(extract_numeric_prefix),
    "domain_knowledge_desc": raw["¿Cuál es tu nivel de conocimiento sobre animales en entornos salvajes?"].apply(extract_label_after_score),
    "preferred_response_length": raw["¿Qué longitud de respuesta prefieres normalmente? "].map(normalize_whitespace),
    "preferred_technical_level": raw["¿Qué nivel técnico prefieres en una explicación? "].map(normalize_whitespace),
    "preferred_explanation_types_raw": raw["¿Qué tipo de explicación te ayuda más a entender una decisión de IA? (Puedes seleccionar varios)"].map(normalize_whitespace),
    "preferred_explanation_types_list": raw["¿Qué tipo de explicación te ayuda más a entender una decisión de IA? (Puedes seleccionar varios)"].apply(safe_json_list),
    "preferred_format": raw["¿Qué formato de explicación prefieres? "].map(normalize_whitespace),
    "main_goals_raw": raw["¿Qué buscas principalmente en una explicación de IA? "].map(normalize_whitespace),
    "main_goals_list": raw["¿Qué buscas principalmente en una explicación de IA? "].apply(safe_json_list),
    "perceived_error_impact": raw["Cuando una IA comete un error en este tipo de casos, ¿qué impacto consideras que tiene? "].map(normalize_whitespace),
    "free_comment": raw["Comentario de mejora sobre el cuestionario o lo que se pase por la cabeza (sobre el tema porfa)"].map(normalize_whitespace),
})

profile_df.head(3)


,user_id,timestamp,age_range,education_level,occupation_raw,occupation_list,ai_knowledge_raw,ai_knowledge_level,ai_knowledge_desc,domain_knowledge_raw,...,domain_knowledge_desc,preferred_response_length,preferred_technical_level,preferred_explanation_types_raw,preferred_explanation_types_list,preferred_format,main_goals_raw,main_goals_list,perceived_error_impact,free_comment
0,U001,2026-03-24 12:11:51.041,18–24,Grado,"Estudiante, Investigador/a (académico)","[""Estudiante"", ""Investigador/a (académico)""]",4 Nivel técnico: he estudiado IA/ML o he hecho...,4,Nivel técnico: he estudiado IA/ML o he hecho p...,3 Intermedio: suelo identificar bastantes y en...,...,Intermedio: suelo identificar bastantes y enti...,Media: explicación breve con algo de detalle,"Intermedio: algunos términos técnicos, pero fá...",Explicaciones basadas en reglas o razonamiento...,"[""Explicaciones basadas en reglas o razonamien...",Texto breve + imagen,Confianza: sentir mayor seguridad en la respue...,"[""Confianza: sentir mayor seguridad en la resp...",Medio: el error podría causar cierta confusión...,Igual para gente más inexperta explicación bre...
1,U002,2026-03-26 17:14:42.568,18–24,Grado,Profesional de otro sector,"[""Profesional de otro sector""]",4 Nivel técnico: he estudiado IA/ML o he hecho...,4,Nivel técnico: he estudiado IA/ML o he hecho p...,"2 Básico: reconozco animales comunes (león, el...",...,"Básico: reconozco animales comunes (león, elef...",Media: explicación breve con algo de detalle,Simple: lenguaje claro y sin tecnicismos,"Explicaciones con ejemplos similares, Explicac...","[""Explicaciones con ejemplos similares"", ""Expl...",Texto detallado + imagen,Transparencia: entender por qué el sistema tom...,"[""Transparencia: entender por qué el sistema t...",Medio: el error podría causar cierta confusión...,NaN
2,U003,2026-03-26 18:02:08.412,18–24,Grado,"Estudiante, Desarrolladora","[""Estudiante"", ""Desarrolladora""]",5 Experto: trabajo/investigo en IA; implemento...,5,"Experto: trabajo/investigo en IA; implemento, ...","2 Básico: reconozco animales comunes (león, el...",...,"Básico: reconozco animales comunes (león, elef...",Corta: solo lo esencial,Técnico: explicación más especializada y precisa,"Explicaciones con ejemplos similares, Explicac...","[""Explicaciones con ejemplos similares"", ""Expl...",Texto breve + imagen,Transparencia: entender por qué el sistema tom...,"[""Transparencia: entender por qué el sistema t...",Alto: el error podría tener consecuencias impo...,NaN


## 4. Construir la base de respuestas por imagen 
- 1 fila por usuario e imagen
- columnas con la opción elegida y las valoraciones


In [9]:

image_blocks = [
    (1, "Imagen 1 de 10 ", "¿Qué tan satisfecho/a estás con la explicación mostrada? ", "Después de ver la explicación, ¿cuánta confianza tienes en la respuesta del sistema? ", "¿La explicación te ayudó a entender la decisión del sistema? "),
    (2, "Imagen 2 de 10 ", "¿Qué tan satisfecho/a estás con la explicación mostrada?  2", "Después de ver la explicación, ¿cuánta confianza tienes en la respuesta del sistema?  2", "¿La explicación te ayudó a entender la decisión del sistema?  2"),
    (3, "Imagen 3 de 10 ", "¿Qué tan satisfecho/a estás con la explicación mostrada?  3", "Después de ver la explicación, ¿cuánta confianza tienes en la respuesta del sistema?  3", "¿La explicación te ayudó a entender la decisión del sistema?  3"),
    (4, "Imagen 4 de 10 ", "¿Qué tan satisfecho/a estás con la explicación mostrada?  4", "Después de ver la explicación, ¿cuánta confianza tienes en la respuesta del sistema?  4", "¿La explicación te ayudó a entender la decisión del sistema?  4"),
    (5, "Imagen 5 de 10 ", "¿Qué tan satisfecho/a estás con la explicación mostrada?  5", "Después de ver la explicación, ¿cuánta confianza tienes en la respuesta del sistema?  5", "¿La explicación te ayudó a entender la decisión del sistema?  5"),
    (6, "Imagen 6 de 10 ", "¿Qué tan satisfecho/a estás con la explicación mostrada?  6", "Después de ver la explicación, ¿cuánta confianza tienes en la respuesta del sistema?  6", "¿La explicación te ayudó a entender la decisión del sistema?  6"),
    (7, "Imagen 7 de 10 ", "¿Qué tan satisfecho/a estás con la explicación mostrada?  7", "Después de ver la explicación, ¿cuánta confianza tienes en la respuesta del sistema?  7", "¿La explicación te ayudó a entender la decisión del sistema?  7"),
    (8, "Imagen 8 de 10 ", "¿Qué tan satisfecho/a estás con la explicación mostrada?  8", "Después de ver la explicación, ¿cuánta confianza tienes en la respuesta del sistema?  8", "¿La explicación te ayudó a entender la decisión del sistema?  8"),
    (9, "Imagen 9 de 10 ", "¿Qué tan satisfecho/a estás con la explicación mostrada?  9", "Después de ver la explicación, ¿cuánta confianza tienes en la respuesta del sistema?  9", "¿La explicación te ayudó a entender la decisión del sistema?  9"),
    (10, "Imagen 10 de 10 ", "¿Qué tan satisfecho/a estás con la explicación mostrada?  10", "Después de ver la explicación, ¿cuánta confianza tienes en la respuesta del sistema?  10", "¿La explicación te ayudó a entender la decisión del sistema?  10"),
    (11, "Extra", "¿Qué tan satisfecho/a estás con la explicación mostrada?  11", "Después de ver la explicación, ¿cuánta confianza tienes en la respuesta del sistema?  11", "¿La explicación te ayudó a entender la decisión del sistema?  11"),
]

records = []
for row_idx in range(len(raw)):
    user_id = profile_df.loc[row_idx, "user_id"]
    for image_id, option_col, sat_col, conf_col, und_col in image_blocks:
        records.append({
            "user_id": user_id,
            "image_id": image_id,
            "image_label": f"img_{image_id:02d}",
            "selected_option": normalize_whitespace(raw.loc[row_idx, option_col]),
            "satisfaction": raw.loc[row_idx, sat_col],
            "confidence": raw.loc[row_idx, conf_col],
            "understanding": raw.loc[row_idx, und_col],
        })

responses_long = pd.DataFrame(records)
responses_long["mean_helpfulness"] = responses_long[["satisfaction", "confidence", "understanding"]].mean(axis=1)
responses_long["case_local_id"] = responses_long.groupby("user_id").cumcount() + 1

responses_long.head(12)


,user_id,image_id,image_label,selected_option,satisfaction,confidence,understanding,mean_helpfulness,case_local_id
0,U001,1,img_01,Opción B,4,5,4,4.333333,1
1,U001,2,img_02,Opción A,4,5,4,4.333333,2
2,U001,3,img_03,Opción E,4,3,4,3.666667,3
3,U001,4,img_04,Opción C,4,4,4,4.000000,4
4,U001,5,img_05,Opción F (Ninguna),2,2,4,2.666667,5
5,U001,6,img_06,Opción D,4,3,4,3.666667,6
6,U001,7,img_07,Opción C,3,2,4,3.000000,7
7,U001,8,img_08,Opción B,4,4,4,4.000000,8
8,U001,9,img_09,Opción E,4,4,4,4.000000,9
9,U001,10,img_10,Opción E,4,4,4,4.000000,10


## 5. Crear la base de casos doble parcial

Aquí unimos el perfil de usuario con la respuesta de cada imagen.

> Esto ya sirve como base de casos doble para recuperación basada en preferencias + valoración de explicación.


In [10]:

case_base_double = responses_long.merge(profile_df, on="user_id", how="left")
case_base_double.insert(0, "case_id", [f"C{idx:04d}" for idx in range(1, len(case_base_double) + 1)])

# Reordenar columnas principales
main_cols = [
    "case_id", "user_id", "image_id", "image_label",
    "selected_option", "satisfaction", "confidence", "understanding", "mean_helpfulness",
    "age_range", "education_level", "occupation_raw",
    "ai_knowledge_level", "domain_knowledge_level",
    "preferred_response_length", "preferred_technical_level",
    "preferred_format", "perceived_error_impact",
    "preferred_explanation_types_raw", "main_goals_raw"
]
other_cols = [c for c in case_base_double.columns if c not in main_cols]
case_base_double = case_base_double[main_cols + other_cols]

print(case_base_double.shape)
case_base_double.head(5)


(297, 30)


,case_id,user_id,image_id,image_label,selected_option,satisfaction,confidence,understanding,mean_helpfulness,age_range,...,case_local_id,timestamp,occupation_list,ai_knowledge_raw,ai_knowledge_desc,domain_knowledge_raw,domain_knowledge_desc,preferred_explanation_types_list,main_goals_list,free_comment
0,C0001,U001,1,img_01,Opción B,4,5,4,4.333333,18–24,...,1,2026-03-24 12:11:51.041,"[""Estudiante"", ""Investigador/a (académico)""]",4 Nivel técnico: he estudiado IA/ML o he hecho...,Nivel técnico: he estudiado IA/ML o he hecho p...,3 Intermedio: suelo identificar bastantes y en...,Intermedio: suelo identificar bastantes y enti...,"[""Explicaciones basadas en reglas o razonamien...","[""Confianza: sentir mayor seguridad en la resp...",Igual para gente más inexperta explicación bre...
1,C0002,U001,2,img_02,Opción A,4,5,4,4.333333,18–24,...,2,2026-03-24 12:11:51.041,"[""Estudiante"", ""Investigador/a (académico)""]",4 Nivel técnico: he estudiado IA/ML o he hecho...,Nivel técnico: he estudiado IA/ML o he hecho p...,3 Intermedio: suelo identificar bastantes y en...,Intermedio: suelo identificar bastantes y enti...,"[""Explicaciones basadas en reglas o razonamien...","[""Confianza: sentir mayor seguridad en la resp...",Igual para gente más inexperta explicación bre...
2,C0003,U001,3,img_03,Opción E,4,3,4,3.666667,18–24,...,3,2026-03-24 12:11:51.041,"[""Estudiante"", ""Investigador/a (académico)""]",4 Nivel técnico: he estudiado IA/ML o he hecho...,Nivel técnico: he estudiado IA/ML o he hecho p...,3 Intermedio: suelo identificar bastantes y en...,Intermedio: suelo identificar bastantes y enti...,"[""Explicaciones basadas en reglas o razonamien...","[""Confianza: sentir mayor seguridad en la resp...",Igual para gente más inexperta explicación bre...
3,C0004,U001,4,img_04,Opción C,4,4,4,4.000000,18–24,...,4,2026-03-24 12:11:51.041,"[""Estudiante"", ""Investigador/a (académico)""]",4 Nivel técnico: he estudiado IA/ML o he hecho...,Nivel técnico: he estudiado IA/ML o he hecho p...,3 Intermedio: suelo identificar bastantes y en...,Intermedio: suelo identificar bastantes y enti...,"[""Explicaciones basadas en reglas o razonamien...","[""Confianza: sentir mayor seguridad en la resp...",Igual para gente más inexperta explicación bre...
4,C0005,U001,5,img_05,Opción F (Ninguna),2,2,4,2.666667,18–24,...,5,2026-03-24 12:11:51.041,"[""Estudiante"", ""Investigador/a (académico)""]",4 Nivel técnico: he estudiado IA/ML o he hecho...,Nivel técnico: he estudiado IA/ML o he hecho p...,3 Intermedio: suelo identificar bastantes y en...,Intermedio: suelo identificar bastantes y enti...,"[""Explicaciones basadas en reglas o razonamien...","[""Confianza: sentir mayor seguridad en la resp...",Igual para gente más inexperta explicación bre...


## 6. Comprobaciones de consistencia

In [12]:

n_users = profile_df["user_id"].nunique()
n_images = responses_long["image_id"].nunique()
n_cases = len(case_base_double)

print("Usuarios únicos:", n_users)
print("Imágenes únicas:", n_images)
print("Casos totales:", n_cases)
print("Esperado:", n_users * n_images)



Usuarios únicos: 27
Imágenes únicas: 11
Casos totales: 297
Esperado: 297


## 7. Versión completa de la base de casos 


In [19]:

# Plantilla
image_metadata_path = OUTPUT_DIR / "image_metadata_template.csv"


# Unión 
image_metadata = pd.read_csv(image_metadata_path)

case_base_double_full = case_base_double.merge(
    image_metadata,
    on=["image_id", "image_label"],
    how="left"
)

case_base_double_full.head(3)


,case_id,user_id,image_id,image_label,selected_option,satisfaction,confidence,understanding,mean_helpfulness,age_range,...,model_predicted_class,model_confidence,initial_description,vqa_support_description,option_A_type,option_B_type,option_C_type,option_D_type,option_E_type,option_F_type
0,C0001,U001,1,img_01,Opción B,4,5,4,4.333333,18–24,...,NaN,NaN,NaN,NaN,Anchor,Grad-CAM,IG,LIME,Saliency,NaN
1,C0002,U001,2,img_02,Opción A,4,5,4,4.333333,18–24,...,NaN,NaN,NaN,NaN,Anchor,Grad-CAM,IG,LIME,Saliency,NaN
2,C0003,U001,3,img_03,Opción E,4,3,4,3.666667,18–24,...,NaN,NaN,NaN,NaN,Anchor,Grad-CAM,IG,LIME,Saliency,NaN


## 8. Guardar salidas

In [20]:

profile_path = OUTPUT_DIR / "user_profiles.csv"
responses_path = OUTPUT_DIR / "image_responses_long.csv"
cases_partial_path = OUTPUT_DIR / "case_base_double_partial.csv"
cases_full_path = OUTPUT_DIR / "case_base_double_full.csv"
jsonl_path = OUTPUT_DIR / "case_base_double_partial.jsonl"

profile_df.to_csv(profile_path, index=False, encoding="utf-8-sig")
responses_long.to_csv(responses_path, index=False, encoding="utf-8-sig")
case_base_double.to_csv(cases_partial_path, index=False, encoding="utf-8-sig")
case_base_double_full.to_csv(cases_full_path, index=False, encoding="utf-8-sig")
case_base_double.to_json(jsonl_path, orient="records", lines=True, force_ascii=False)

print("Archivos generados:")
for p in [profile_path, responses_path, cases_partial_path, cases_full_path, image_metadata_path, jsonl_path]:
    print("-", p)


Archivos generados:
- cbr_case_base_outputs/user_profiles.csv
- cbr_case_base_outputs/image_responses_long.csv
- cbr_case_base_outputs/case_base_double_partial.csv
- cbr_case_base_outputs/case_base_double_full.csv
- cbr_case_base_outputs/image_metadata_template.csv
- cbr_case_base_outputs/case_base_double_partial.jsonl


/var/folders/hb/_dcyvg9n4j52gmgfrylw33wc0000gn/T/ipykernel_60058/2546114055.py:11: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  case_base_double.to_json(jsonl_path, orient="records", lines=True, force_ascii=False)


## 9. Estado del CBR


### Problema (`problem`)
- capa de contenido
  - `image`
  - `description_xai`
  - `image_domain`
  - `class`
- perfil del usuario:
  - `ai_knowledge_level`
  - `domain_knowledge_level`
  - `preferred_response_length`
  - `preferred_technical_level`
  - `preferred_format`
  - `preferred_explanation_types`
  - `main_goals`


### Solución (`solution`)
- `selected_option`
- `selected_xai_method`
- `mean_helpfulness`
- `response_length`
- `technical_level`
- `explanation_type`
- `undersoutput_formattanding`


